In [1]:
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    roc_curve,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

import sys
import os

sys.path.append(os.path.abspath('..'))

from src import final_models , get_preprocessor, loader 

df =  loader()
preprocessor = get_preprocessor() 
models = final_models(preprocessor) 



X , Y = df.drop(columns = ['Churn']) , df['Churn'].map({'Yes' : 1 , 'No' : 0})

x_train , x_test , y_train , y_test = train_test_split(X , Y , test_size = 0.2 , random_state = 42)


In [2]:
result = {}

for model_name , model in models.items():
    model.fit(x_train , y_train)
    predictions = model.predict(x_test)

    result[model_name] = {
        'accuracy' : accuracy_score(y_test , predictions),
        'precision' : precision_score(y_test , predictions),
        'recall' : recall_score(y_test , predictions),
        'f1_score' : f1_score(y_test , predictions),
        'roc_auc' : roc_auc_score(y_test , predictions)
    }


/home/fangyuan/Projects/telco-customer-segmentation-churn/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


In [3]:
import pandas as pd
result_df = pd.DataFrame(columns = ['accuracy' , 'precision' , 'recall' , 'f1_score' , 'roc_auc'] , index = result.keys() , data = result.values())
result_df.sort_values(by = 'recall' , ascending = False , inplace = True)
result_df

,accuracy,precision,recall,f1_score,roc_auc
logistic_regression,0.756565,0.525424,0.831099,0.643821,0.780414
SVC,0.770050,0.547389,0.758713,0.635955,0.766422
XGBoost,0.788502,0.603306,0.587131,0.595109,0.724068
random_forest,0.779276,0.588068,0.554960,0.571034,0.707499
decision_tree,0.716111,0.468822,0.544236,0.503722,0.661114


In [4]:
model = models['logistic_regression'] 

param_grid = {
    "classifier__solver": ["saga"],
    "classifier__penalty": ["elasticnet"],
    "classifier__C": [0.01, 0.1, 1, 10],
    "classifier__l1_ratio": [1, 0.5, 0],
    "classifier__max_iter": [1000 , 1500, 2000 , 2500, 3000],
    "classifier__class_weight": [None, "balanced"],
}

grid_search = RandomizedSearchCV(
    model,
    param_distributions=param_grid,
    n_iter=50,
    scoring="recall",
    cv=5,
    random_state=42,
    n_jobs=-1,
)



In [5]:
model = grid_search.fit(x_train , y_train).best_estimator_ 

/home/fangyuan/Projects/telco-customer-segmentation-churn/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/fangyuan/Projects/telco-customer-segmentation-churn/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf in

In [6]:
grid_search.best_params_

{'classifier__solver': 'saga',
 'classifier__penalty': 'elasticnet',
 'classifier__max_iter': 2000,
 'classifier__l1_ratio': 1,
 'classifier__class_weight': 'balanced',
 'classifier__C': 10}

In [7]:
predictions = model.predict(x_test)

result_final = {
    'name' : 'logistic_regression',
    'accuracy' : accuracy_score(y_test , predictions),
    'precision' : precision_score(y_test , predictions , pos_label=1),
    'recall' : recall_score(y_test , predictions , pos_label=1),
    'f1_score' : f1_score(y_test , predictions, pos_label=1),
    'roc_auc' : roc_auc_score(y_test , predictions) 
}

In [8]:
result_final = pd.DataFrame(result_final, index = [0])
result_final

,name,accuracy,precision,recall,f1_score,roc_auc
0,logistic_regression,0.753016,0.521008,0.831099,0.640496,0.778001
